# Phase 2 - Choose the target lakes, and measure the ceiling

Two outputs.

**The gate.** One large Adirondack lake and one small one, both with enough July-years
to support a ~30-point annual correlation. If no small lake qualifies, `select_target_lakes`
raises. That is the intended behaviour: the small-lake case is the hard one, and finding
out now costs an afternoon.

**The ceiling.** The intraclass correlation of log Secchi across the region's lakes. It is
the fraction of the pooled signal that says nothing about tracking one lake through time,
and it is the number that reconciles a published R-squared of 0.637 with a per-lake r of
-0.22.

In [ ]:
# Cell 1 of 5: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity-adirondack.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity-adirondack")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

In [ ]:
# Cell 2 of 5: region subset
import pandas as pd
from lakeclarity import config, eda, edi, features, locus, selection, viz

viz.use_style()

matchups = edi.load_matchups()
lakes = locus.load_lakes()
adk = locus.adirondack_lakes(lakes)
print(f"{len(adk):,} Adirondack lakes in LOCUS")

region = matchups[matchups["lagoslakeid"].isin(adk["lagoslakeid"])].copy()
region = region[region[config.TARGET].notna()]
region = features.add_time_columns(region)
region = features.add_log_target(region)
print(f"{len(region):,} Secchi matchups on {region.lagoslakeid.nunique():,} Adirondack lakes")
region.to_parquet(config.INTERIM_DIR / "region_matchups.parquet", index=False)

In [ ]:
# Cell 3 of 5: THE CEILING. Run this before choosing anything.
ceiling = selection.ceiling_report(region)
for k, v in ceiling.items():
    print(f"{k:>28}: {v:,.3f}")

print("\nRead this as: of all the variation in lake clarity across the Adirondacks,")
print(f"{ceiling['pct_variance_between_lakes']:.0f}% is lakes differing from one another and only")
print(f"{ceiling['pct_variance_within_lakes']:.0f}% is any given lake changing across the years.")
print("\nA model that learns the first and none of the second scores a superb pooled R2")
print("and cannot track a single lake. That is the national model's failure mode.")

pd.Series(ceiling).to_csv(config.TABLE_DIR / "T03_variance_ceiling.csv")

In [ ]:
# Cell 4 of 5: the gate
candidates = selection.candidate_table(region, adk)
candidates.to_csv(config.TABLE_DIR / "T04_candidate_lakes.csv", index=False)
print(candidates.head(15)[["lagoslakeid", "lake_name", "lake_waterarea_ha",
                            "n_matchups", "n_july", "n_july_years",
                            "secchi_mean", "secchi_sd", "pixelcount_median"]].to_string(index=False))

try:
    targets = selection.select_target_lakes(candidates)
    print("\n" + targets.summary())
    pd.Series({"large": targets.ids[0], "small": targets.ids[1]}).to_csv(
        config.TABLE_DIR / "T05_target_lakes.csv")
except selection.NoEligibleLakeError as exc:
    print("\nGATE FAILED:", exc)
    print("\nDo not work around this. Either relax MIN_JULY_MATCHUPS with a written")
    print("justification, widen the region beyond the Adirondack counties, or accept")
    print("that the small-lake case cannot be validated with the data that exists.")
    raise

In [ ]:
# Cell 5 of 5: figures F6 to F9
for fig, fid, slug in [
    (selection.fig_region_map(candidates, targets), "F06", "region_map"),
    (selection.fig_area_vs_pixels(candidates, targets), "F07", "area_vs_pixels"),
    (selection.fig_variance_decomposition(region), "F08", "variance_decomposition"),
    (selection.fig_candidate_timeseries(region, candidates), "F09", "candidate_timeseries"),
]:
    print("wrote", viz.save(fig, fid, slug).name)